In [1]:
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

index = load_index_from_storage(StorageContext.from_defaults(persist_dir=r"balakanda_chapter1_update"))


[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\conda_tmp\ipykernel_10860\4129441308.py:7: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


In [2]:
query_engine1 = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)

In [4]:
response = query_engine1.query("Name rama's bow that belonged to indra?")
print(response)

Prompt num:  3
Extracted keywords: ['Rama', 'Indra', 'bow']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('rama', 'received_bow_sword_quivers', 'agastya')
('rama', 'requested_by_viswamitra', 'view_bow')
('rama', "broke_shiva's_bow", 'canto_68')
('rama', 'context', 'bow_sacrifice')
('rama', 'broke', "shiva's_bow")
Prompt num:  4
The passages state that Rama received a divine bow, sword, and quivers from Agastya, and these items were originally given to Agastya by Indra.  Therefore, the bow Rama received that belonged to Indra doesn't have a specific name mentioned in the provided text.



In [ ]:
# storage_context2 = StorageContext.from_defaults(persist_dir=r"C:\Users\M LAKSHMI NARASIMHA\OneDrive\Desktop\Ramayan\KGs\yuddhakanda")
# indexi = load_index_from_storage(storage_context2)
import json
with open('parsed_sundarakanda_triplets.json', 'r') as file:
    parsed_data = json.load(file)

file = 'sundarakanda.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['content'],
    metadata={
        "canto": i['chapter'],
        "canto_summary": i['summary'],
        "r_id": str(i['chapter']) + "_" + str(i['prompt_number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents), len(parsed_data))
print(f"Document Metadata: {documents[-1].metadata}")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import KnowledgeGraphIndex

node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents[:3])
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex([], 
                            storage_context=storage_context, 
                            include_embeddings=True)
from tqdm import tqdm
for i in tqdm(range(0, len(nodes))):
    if i > 0 and i % 30 == 0:
        index.storage_context.persist(persist_dir='yuddhakanda_temp')
    for key,value in parsed_data[i].items():
        for tup in value:
            index.upsert_triplet_and_node(tuple(tup), nodes[i], include_embeddings=True)

query_engine1 = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
# response = query_engine1.query("who is demon?")

In [ ]:
query_engine1.retriever.keywords = 

In [ ]:
from gg import *

In [ ]:
p = query_engine1.retrieve("who is demon?")

In [ ]:
g = post_process(p)

In [ ]:
print(g)

In [ ]:
p

In [ ]:
query_index("who is demon?")

In [ ]:
p[-1].metadata

In [ ]:
print(p[-1].text)

In [ ]:
t = [1,2,3,4,5,6,7,8,9,10]
t[:]